In [1]:
import json
import os
import subprocess
import sys
import threading
from pathlib import Path

import py3Dmol
from sample.sample_config import (
    GenerationParams,
    SampleCheckpointParams,
    SampleConfig,
    SampleOutputParams,
)
from train.train_config import (
    CheckpointParams,
    TrainLoaderConfig,
    TrainConfig,
    TrainingParams,
)

PALLATOM_ROOT = Path.cwd().resolve()
if str(PALLATOM_ROOT) not in sys.path:
    sys.path.insert(0, str(PALLATOM_ROOT))

PALLATOM_ROOT


PosixPath('/workspaces/diffusion/pallatom')

In [2]:
import torch

torch.cuda.is_available()

True

# PallAtom: Training & Analysis

1. **Configure** — tune hyperparameters and serialise `TrainConfig` to JSON
2. **Train** — launch `train_loop.py` as a subprocess
3. **Sample** — load the saved checkpoint and run EDM backbone sampling
4. **Visualise** — render sampled structures with py3Dmol

## 1 · Training Configuration

In [3]:
def run_subprocess(cmd, out: dict):
    env = os.environ.copy()
    env["PYTHONPATH"] = str(PALLATOM_ROOT)
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd=str(PALLATOM_ROOT),
        env=env,
    )
    out["pid"] = proc.pid
    print(f"Subprocess PID: {proc.pid}")

    stdout, _ = proc.communicate()
    output_lines = stdout.strip().splitlines() if stdout else []

    if proc.returncode != 0:
        print([f"ERROR (exit {proc.returncode}):"] + output_lines)
        out["result"] = None
        return

    out["result"] = output_lines

In [4]:
tcfg = TrainConfig(
    training=TrainingParams(
        num_epochs=50,
        # pretrained_weights="pallatom_toy_best.pt"
        ),
    checkpoint=CheckpointParams(checkpoint_path="pallatom_toy_best.pt"),
    train_loader=TrainLoaderConfig(max_seq_length=128)
    )
tcfg

TrainConfig(training=TrainingParams(num_epochs=50, lr=0.001, weight_decay=0.0001, grad_clip=10.0, pretrained_weights=None, resume_checkpoint=None, accumulated_token_budget=4096, lr_decay_steps=50000, lr_decay_factor=0.95), model=ModelParams(window_size=32, f_ref_dim=35, c_atom=128, c_pair=128, c_res=256, c_atompair=16, K_unit=8, max_residues=128, n_amino=20, n_blocks_atom_transformer_encoder=3, n_heads_atom_transformer_encoder=4, n_blocks_atom_transformer_decoder=3, n_heads_atom_transformer_decoder=4, n_pairformer_blocks_template_embedder=2, n_paiformer_heads_template_embedder=16), noise=NoiseScheduleParams(sigma_data=16.0, sigma_max=160, sigma_min=0.0004, P_mean=-1.2, P_std=1.5), distogram_res=ResidueDistogramParams(min_dist=3.25, max_dist=50.75, n_bins=39, tok_emb_dim=32), distogram_atom=AtomDistogramParams(min_dist=0.0, max_dist=10.0, n_bins=22, tok_emb_dim=32), loss=LossParams(lam=1.0, alpha_0=0.25, alpha_1=1.0, alpha_2=0.5, alpha_3=0.5, alpha_4=1.0, gamma=0.99, smooth_lddt_cutoff=

In [5]:
config_json_path = PALLATOM_ROOT / "train" / "run_config.json"
_ = config_json_path.write_text(
    tcfg.model_dump_json(indent=2)
)

In [ ]:
log_path = PALLATOM_ROOT / "train" / "train_logs.jsonl"
shard_dir: Path = PALLATOM_ROOT / "data" / "shards"

train_cmd = [
            sys.executable, "-u",
            str(PALLATOM_ROOT / "train" / "train_loop.py"),
            "--dataset_jsonl",        str(PALLATOM_ROOT / "data" / "chain_set.jsonl"),
            "--keys_for_splits_json",      str(PALLATOM_ROOT / "data" / "chain_set_splits.json"),
            "--config",      config_json_path,
            "--structlog_jsonl",    log_path,
            # "--debug_run",
            "--shard_dir", shard_dir,
        ]

training_out = dict()
threading.Thread(
    target=run_subprocess,
    args=(train_cmd, training_out),
    daemon=True
).start()

Subprocess PID: 5268


['ERROR (exit -15):', 'wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.', 'wandb: Currently logged in as: tlmattesonr to https://api.wandb.ai. Use `wandb login --relogin` to force relogin', 'wandb: Tracking run with wandb version 0.26.1', 'wandb: Run data is saved locally in /workspaces/diffusion/pallatom/wandb/run-20260701_214842-dl11w7sl', 'wandb: Run `wandb offline` to turn off syncing.', 'wandb: Syncing run quiet-dust-260', 'wandb: ⭐️ View project at https://wandb.ai/tlmattesonr/pallatom-training', 'wandb: 🚀 View run at https://wandb.ai/tlmattesonr/pallatom-training/runs/dl11w7sl', "\x1b2026-07-01T21:48:43.758112Z\x1b [\x1b\x1binfo     \x1b[0m] \x1btraining                      \x1b \x1bddp\x1b=\x1bFalse\x1b \x1bdevice\x1b=\x1bdevice(type='cuda', index=0)\x1b", '\x1b2026-07-01T21:48:43.759231Z\x1b [\x1b\x1binfo     \x1b[0m] \x1bgradient_accumulation         \x1b \x1bglobal_token_budget\x1b=\x1b4096\x1b \x1btoken_budget_per_rank\x1b=\x1b4096\x1b'

In [ ]:
import torch

NUM_GPUS = torch.cuda.device_count()
torchrun = str(Path(sys.executable).parent / "torchrun")
ddp_log_path = PALLATOM_ROOT / "train" / "train_logs_ddp.jsonl"
shard_dir: Path = PALLATOM_ROOT / "data" / "shards"

ddp_train_cmd = [
    torchrun,
    f"--nproc_per_node={NUM_GPUS}",
    str(PALLATOM_ROOT / "train" / "train_loop.py"),
    "--dataset_jsonl",        str(PALLATOM_ROOT / "data" / "chain_set.jsonl"),
    "--keys_for_splits_json",      str(PALLATOM_ROOT / "data" / "chain_set_splits.json"),
    "--config",      config_json_path,
    "--structlog_jsonl",    ddp_log_path,
    # "--debug_run",
    "--shard_dir", shard_dir,
    "--ddp",
]

ddp_training_out = dict()
threading.Thread(
    target=run_subprocess,
    args=(ddp_train_cmd, ddp_training_out),
    daemon=True
).start()

Subprocess PID: 36834


In [3]:
import sys
from pathlib import Path

str(Path(sys.executable).parent / "torchrun")

'/opt/venv/bin/torchrun'

In [ ]:
dist.is_initialized()

In [6]:
ckpt_path       = str(PALLATOM_ROOT / tcfg.checkpoint.checkpoint_path)
ckpt_path

'/workspaces/diffusion/pallatom/pallatom_toy_best.pt'

In [7]:
from sample.sample_config import SamplerParams


sample_output_path     = str(Path(config_json_path).with_name("samples.json"))
sample_cfg_path = str(Path(config_json_path).with_name("sample_config.json"))

scfg = SampleConfig(
    model=tcfg.model,
    noise=tcfg.noise,
    generation=GenerationParams(
        n_res=tcfg.test_loader.max_seq_length,
        n_samples=tcfg.test_loader.batch_size,

    ),
    sampler = SamplerParams(S_noise=1.0),
    checkpoint=SampleCheckpointParams(checkpoint_path=ckpt_path),
    output=SampleOutputParams(output_path=sample_output_path),
)

sample_config_json_path = PALLATOM_ROOT / "train" / "sample_config.json"
with open(sample_config_json_path, "w") as _f:
    _f.write(json.dumps(scfg.model_dump(), indent=2))

In [8]:
sample_log_path = PALLATOM_ROOT / "train" / "sample_logs.jsonl"
sample_cmd = [
            sys.executable, "-u",
            str(PALLATOM_ROOT / "sample" / "sampling.py"),
            "--config", sample_cfg_path,
            "--log_file", sample_log_path,
        ]
# does this not pipe logs out into a structlog
sample_out = dict()
threading.Thread(
    target=run_subprocess,
    args=(sample_cmd, sample_out),
    daemon=True
).start()

Subprocess PID: 913863


In [14]:
sample_out

{'pid': 913863,
 'result': ["\x1b2026-06-29T06:46:54.097836Z\x1b [\x1b\x1binfo     \x1b[0m] \x1bconfig loaded                 \x1b \x1bconfig\x1b=\x1bPosixPath('/workspaces/diffusion/pallatom/train/sample_config.json')\x1b \x1bn_res\x1b=\x1b128\x1b \x1bn_samples\x1b=\x1b8\x1b",
  '\x1b2026-06-29T06:46:56.449833Z\x1b [\x1b\x1binfo     \x1b[0m] \x1bmodel loaded                  \x1b \x1bcheckpoint\x1b=\x1b/workspaces/diffusion/pallatom/pallatom_toy_best.pt\x1b \x1bdevice\x1b=\x1bcuda\x1b',
  '\x1b2026-06-29T06:46:56.799349Z\x1b [\x1b\x1binfo     \x1b[0m] \x1bsampling                      \x1b \x1bddim_steps\x1b=\x1b200\x1b \x1bn_res\x1b=\x1b128\x1b \x1bn_samples\x1b=\x1b8\x1b',
  '/workspaces/diffusion/pallatom/helpers/alignment.py:312: UserWarning: torch.qr is deprecated in favor of torch.linalg.qr and will be removed in a future PyTorch release.',
  "The boolean parameter 'some' has been replaced with a string parameter 'mode'.",
  'Q, R = torch.qr(A, some)',
  'should be replaced with

In [16]:
# lets load the pdb files from samples.json

# Open the file in read mode ('r')
with open('train/samples.json') as file:
    # Use json.load() to read and parse the file
    data = json.load(file)

# Now 'data' is a standard Python object (dict or list)
print(len(data))
pdb_str = data[0]
view = py3Dmol.view(
    width=600, height=600, linked=True , viewergrid=(1, 1))
view.setViewStyle({'style': 'outline', 'color': 'black', 'width': 0.1})
style = {"cartoon": {'color': 'spectrum'}}

view.addModelsAsFrames(pdb_str, viewer=(0, 0))
view.setStyle({'model': -1}, style, viewer=(0, 0))
view.zoomTo(viewer=(0, 0))

view.render()


# and then show them in py3dmol

8


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Sample from the API

In [ ]:
import time
import requests

REST_APIS_ROOT = PALLATOM_ROOT.parent / "REST_APIs"
API_HOST = "127.0.0.1"
API_PORT = 8000
API_BASE = f"http://{API_HOST}:{API_PORT}"

# Start the server only if not already running
try:
    requests.get(f"{API_BASE}/health", timeout=1)
    print("Server already running")
except requests.exceptions.ConnectionError:
    env = {**os.environ, "PYTHONPATH": str(PALLATOM_ROOT), "CHECKPOINT_PATH": ckpt_path}
    api_proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "api:app", "--host", API_HOST, "--port", str(API_PORT)],
        cwd=str(REST_APIS_ROOT),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(f"Server started (PID {api_proc.pid}), waiting for model to load...")

    for _ in range(120):
        time.sleep(1)
        try:
            if requests.get(f"{API_BASE}/health", timeout=2).ok:
                print("Server ready")
                break
        except requests.exceptions.ConnectionError:
            pass
    else:
        raise RuntimeError("Server did not become ready within 120 s")

# Unconditionally Sample
resp = requests.post(
    f"{API_BASE}/sample",
    json={
        "n_res": scfg.generation.n_res,
        "n_samples": scfg.generation.n_samples,
        "ddim_steps": scfg.sampler.ddim_steps,
    },
    timeout=300,
)
resp.raise_for_status()
payload = resp.json()
pdb_strings = payload["pdb_strings"]
print(f"Received {len(pdb_strings)} structures ({payload['n_res']} res each) from {payload['device']}")


conditional sampling from amino acid sequence alone (no templates)

In [ ]:
n_res = scfg.generation.n_res
AA20 = "ACDEFGHIKLMNPQRSTVWY"
example_seq = (AA20 * (n_res // len(AA20) + 1))[:n_res]

resp = requests.post(
    f"{API_BASE}/sample",
    json={
        "n_res": n_res,
        "n_samples": scfg.generation.n_samples,
        "ddim_steps": scfg.sampler.ddim_steps,
        "sequence": example_seq,
    },
    timeout=300,
)
resp.raise_for_status()
seq_cond_pdbs = resp.json()["pdb_strings"]
print(f"[seq only] {len(seq_cond_pdbs)} structures")


sequence + partial template

In [ ]:
def slice_pdb(pdb_str: str, n_residues: int) -> str:
    """Return a PDB string truncated to the first n_residues unique residue numbers."""
    seen: list[int] = []
    kept: list[str] = []
    for line in pdb_str.splitlines():
        if line.startswith("ATOM"):
            res_num = int(line[22:26])
            if res_num not in seen:
                seen.append(res_num)
            if len(seen) > n_residues:
                continue
        kept.append(line)
    return "\n".join(kept)

partial_pdb = slice_pdb(pdb_strings[0], n_res // 2)

resp = requests.post(
    f"{API_BASE}/sample",
    json={
        "n_res": n_res,
        "n_samples": scfg.generation.n_samples,
        "ddim_steps": scfg.sampler.ddim_steps,
        "sequence": example_seq,
        "template_pdb": partial_pdb,
    },
    timeout=300,
)
resp.raise_for_status()
seq_partial_templ_pdbs = resp.json()["pdb_strings"]
print(f"[seq + partial template] {len(seq_partial_templ_pdbs)} structures")


no sequence + partial template

In [ ]:
resp = requests.post(
    f"{API_BASE}/sample",
    json={
        "n_res": n_res,
        "n_samples": scfg.generation.n_samples,
        "ddim_steps": scfg.sampler.ddim_steps,
        "template_pdb": partial_pdb,
    },
    timeout=300,
)
resp.raise_for_status()
partial_templ_pdbs = resp.json()["pdb_strings"]
print(f"[partial template only] {len(partial_templ_pdbs)} structures")

no sequence + full template

In [ ]:
resp = requests.post(
    f"{API_BASE}/sample",
    json={
        "n_res": n_res,
        "n_samples": scfg.generation.n_samples,
        "ddim_steps": scfg.sampler.ddim_steps,
        "template_pdb": pdb_strings[0],
    },
    timeout=300,
)
resp.raise_for_status()
full_templ_pdbs = resp.json()["pdb_strings"]
print(f"[full template only] {len(full_templ_pdbs)} structures")
